# Notebook 01 — Residuals Are Not Noise

**Repo:** `residual-phase-lock`  
**Notebook:** `01_residual_is_not_noise.ipynb`

## Claim

> Residuals are not automatically noise. Residuals can reveal hidden structure.

This notebook builds a controlled signal with a known hidden structural component, fits an intentionally incomplete baseline model, and shows that the residual preserves coherent structure.

## Core compression

```text
model fit ≠ structure exhausted
residual ≠ noise
residual → hidden structure signal
phase-lock begins by measuring residual structure
```

## 1. Setup

This notebook uses standard scientific Python packages and runs cleanly in Google Colab.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_current_figure(name):
    path = os.path.join(FIG_DIR, f"{name}.png")
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print(f"Saved figure: {path}")

def save_json(obj, name):
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"Saved JSON: {path}")

## 2. Generate a signal with hidden structure

The toy signal has three parts:

1. a simple linear trend,
2. a structured oscillatory component,
3. small random noise.

The baseline model will fit only the linear trend. The residual should therefore contain the hidden structure.

In [ ]:
n = 500
x = np.linspace(0, 10, n)

trend = 0.8 * x + 1.5
hidden_structure = 0.9 * np.sin(2.5 * x) + 0.35 * np.sin(6.0 * x)
noise = np.random.normal(0, 0.25, size=n)

y = trend + hidden_structure + noise
X = x.reshape(-1, 1)

data = pd.DataFrame({
    "x": x,
    "observed_y": y,
    "true_trend": trend,
    "hidden_structure": hidden_structure,
    "noise": noise,
})
data.to_csv(os.path.join(RESULTS_DIR, "01_toy_signal_data.csv"), index=False)

data.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.5)
plt.plot(x, trend, label="true linear trend", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Toy signal: trend + hidden structure + noise")
plt.legend()
plt.tight_layout()
save_current_figure("01_toy_signal")
plt.show()

## 3. Fit an incomplete baseline model

We deliberately fit only a linear model. This creates a useful failure mode: the model captures trend but leaves structured residuals behind.

In [ ]:
model = LinearRegression()
model.fit(X, y)
y_hat = model.predict(X)

residual = y - y_hat

rmse = float(np.sqrt(mean_squared_error(y, y_hat)))
r2 = float(r2_score(y, y_hat))
slope = float(model.coef_[0])
intercept = float(model.intercept_)

baseline_metrics = {
    "baseline_rmse": rmse,
    "baseline_r2": r2,
    "fitted_slope": slope,
    "fitted_intercept": intercept,
}

print(f"Baseline linear RMSE: {rmse:.4f}")
print(f"Baseline linear R²:   {r2:.4f}")
print(f"Fitted slope:         {slope:.4f}")
print(f"Fitted intercept:     {intercept:.4f}")

save_json(baseline_metrics, "01_baseline_metrics")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.5)
plt.plot(x, y_hat, label="linear fit", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Incomplete baseline fit")
plt.legend()
plt.tight_layout()
save_current_figure("01_incomplete_baseline_fit")
plt.show()

## 4. Inspect the residual

If the residual were merely random noise, it should look unstructured. Instead, coherent oscillatory organization remains.

In [ ]:
residual_df = pd.DataFrame({
    "x": x,
    "residual": residual,
    "hidden_structure": hidden_structure,
})
residual_df.to_csv(os.path.join(RESULTS_DIR, "01_residuals.csv"), index=False)

plt.figure(figsize=(10, 4))
plt.plot(x, residual, label="residual", linewidth=1.5)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("residual")
plt.title("Residual after incomplete fit")
plt.legend()
plt.tight_layout()
save_current_figure("01_residual_after_fit")
plt.show()

## 5. Compare residual to true hidden structure

In real data, hidden structure is usually unknown. Here, because the signal is controlled, we can directly verify that the residual tracks the hidden component.

In [ ]:
corr = float(np.corrcoef(residual, hidden_structure)[0, 1])

plt.figure(figsize=(10, 4))
plt.plot(x, hidden_structure, label="true hidden structure", linewidth=2)
plt.plot(x, residual, label="model residual", linewidth=1.5, alpha=0.85)
plt.xlabel("x")
plt.ylabel("value")
plt.title(f"Residual tracks hidden structure (correlation = {corr:.3f})")
plt.legend()
plt.tight_layout()
save_current_figure("01_residual_vs_hidden_structure")
plt.show()

print(f"Correlation(residual, hidden structure): {corr:.4f}")

## 6. Frequency-domain check

A random residual should not contain sharp organized spectral peaks. The residual spectrum below shows concentrated modes from hidden structure.

In [ ]:
residual_centered = residual - residual.mean()
freqs = np.fft.rfftfreq(n, d=(x[1] - x[0]))
spectrum = np.abs(np.fft.rfft(residual_centered))

spectrum_df = pd.DataFrame({
    "frequency": freqs,
    "amplitude": spectrum,
})
spectrum_df.to_csv(os.path.join(RESULTS_DIR, "01_residual_spectrum.csv"), index=False)

plt.figure(figsize=(10, 4))
plt.plot(freqs, spectrum, linewidth=1.5)
plt.xlabel("frequency")
plt.ylabel("amplitude")
plt.title("Residual spectrum: structured peaks remain")
plt.xlim(0, 2)
plt.tight_layout()
save_current_figure("01_residual_spectrum")
plt.show()

top_indices = np.argsort(spectrum)[-5:][::-1]
dominant = pd.DataFrame({
    "rank": np.arange(1, 6),
    "frequency": freqs[top_indices],
    "amplitude": spectrum[top_indices],
})

dominant.to_csv(os.path.join(RESULTS_DIR, "01_dominant_residual_modes.csv"), index=False)
dominant

## 7. Residual-structure score

We define a simple score:

```text
residual_structure_score = dominant spectral energy / total spectral energy
```

Larger values indicate that residual energy is concentrated in organized modes rather than spread randomly.

In [ ]:
total_energy = float(np.sum(spectrum**2))
dominant_energy = float(np.sum(spectrum[top_indices[:3]]**2))
residual_structure_score = float(dominant_energy / total_energy)

print(f"Residual structure score: {residual_structure_score:.4f}")

## 8. Phase-lock framing

Notebook 01 establishes the detection layer:

```text
residual → structure signal
```

Later notebooks can add the revise loop:

```text
residual → detect mismatch
phase-lock → enforce alignment
revise → improve stability
```

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "baseline_rmse",
        "baseline_r2",
        "fitted_slope",
        "fitted_intercept",
        "residual_hidden_structure_correlation",
        "residual_structure_score",
        "dominant_frequency_1",
        "dominant_frequency_2",
        "dominant_frequency_3",
    ],
    "value": [
        rmse,
        r2,
        slope,
        intercept,
        corr,
        residual_structure_score,
        float(freqs[top_indices[0]]),
        float(freqs[top_indices[1]]),
        float(freqs[top_indices[2]]),
    ],
})

summary.to_csv(os.path.join(RESULTS_DIR, "01_summary.csv"), index=False)

summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
save_json(summary_json, "01_summary")

summary

## 9. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
01_residual_is_not_noise_outputs.zip
├── figures/
└── results/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "01_residual_is_not_noise_outputs.zip"

# Re-confirm folders exist
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Zip output directories
!zip -r $ZIP_NAME figures results

# Download in Google Colab
try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 10. Takeaway

This notebook supports the repo's opening claim:

```text
residuals are not noise by default
residuals can reveal hidden structure
phase-lock begins by measuring residual structure
```

Suggested next notebook:

```text
02_topology_failure_demo.ipynb
```